In [1]:
import os
import sys
import logging

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)

os.environ["CUDA_VISIBLE_DEVICES"] = "3"
#os.environ["TRITON_DEBUG"] = "1"
#os.environ["FLASH_ATTENTION_TRITON_AMD_DEBUG"] = "1"
os.environ["FLASH_ATTENTION_TRITON_AMD_AUTOTUNE"] = os.environ[
    "TRITON_PRINT_AUTOTUNING "] = "0"
#sys.path.append(os.path.realpath('.'))

import math
import time
from datetime import timedelta
import torch
from torch import nn
from tqdm import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, OPTForCausalLM

from torchtitan.float8 import Float8Handler
from torchtitan.config_manager import JobConfig
from torchtitan.parallelisms import (
    models_parallelize_fns,
    models_pipelining_fns,
    ParallelDims,
)
from torchtitan.checkpoint import CheckpointManager, TrainState
from torchtitan.optimizer import build_lr_schedulers, build_optimizers
from torchtitan.datasets import build_hf_data_loader
from torchtitan.logging import init_logger, logger
from torchtitan.models import model_name_to_cls, model_name_to_tokenizer, models_config
from torchtitan.metrics import build_device_memory_monitor, build_metric_logger
from torchtitan.profiling import maybe_enable_memory_snapshot, maybe_enable_profiling
from torchtitan.utils import (device_module, device_type, get_peak_flops,
                              get_train_context, GarbageCollection, Color,
                              NoColor, create_context_parallel_ctx,
                              set_determinism, clip_grad_norm_, dist_mean,
                              dist_max, get_num_params, get_num_flop_per_token,
                              set_pg_timeouts, reset_params)

#device = torch.device(f"{device_type}:{int(os.environ['LOCAL_RANK'])}")
device = torch.device(f"{device_type}:0")
device_module.set_device(device)
device_memory_monitor = build_device_memory_monitor()
gpu_peak_flops = get_peak_flops(device_memory_monitor.device_name)
logger.info(f"Peak FLOPS used for computing MFU: {gpu_peak_flops:.3e}")

# data_path = "/workspace/datasets/wikitext-2-raw-v1/test"
model_path = "/workspace/hf_home/hub/models--facebook--opt-125m/snapshots/27dcfa74d334bc871f3234de431e71c6eeba5dd6"
# #model_path = "/group/ossmodelzoo/sequence_learning/weights/nlp-pretrained-model/llama-2-7b-hf-bf16"

#attn_implementation = "eager"
attn_implementation = "flash_attention_2"
model_config = AutoConfig.from_pretrained(
    model_path,
    attn_implementation=attn_implementation,
    use_cache=False,
)
model = OPTForCausalLM(model_config)

try:
    tokenizer = AutoTokenizer.from_pretrained(model_path,
                                              legacy=False,
                                              use_fast=False)
except:
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)

original_encode_func = tokenizer.encode
def new_encode_func(*args, **kwargs):
    bos = kwargs.pop("bos")
    eos = kwargs.pop("eos")
    t = original_encode_func(*args, **kwargs)
    if bos:
        t.insert(0, tokenizer.bos_token_id)
    if eos:
        t.append(tokenizer.eos_token_id)
    return t


tokenizer.encode = new_encode_func

job_config = JobConfig()
job_config.parse_args(["--job.config_file", "./train_configs/debug_model.toml"])
#job_config.parse_args(["--job.config_file", "./train_configs/opt_125m_fp8.toml"])

#world_size = int(os.environ["WORLD_SIZE"])
world_size = 1

parallel_dims = ParallelDims(
    dp_shard=job_config.training.data_parallel_shard_degree,
    dp_replicate=job_config.training.data_parallel_replicate_degree,
    cp=job_config.experimental.context_parallel_degree,
    tp=job_config.training.tensor_parallel_degree,
    pp=job_config.experimental.pipeline_parallel_degree,
    world_size=world_size,
    enable_loss_parallel=not job_config.training.disable_loss_parallel,
)

color = NoColor if job_config.metrics.disable_color_printing else Color

# build meshes
world_mesh = None
if parallel_dims.dp_enabled:
    dp_mesh = world_mesh["dp"]
    dp_degree, dp_rank = dp_mesh.size(), dp_mesh.get_local_rank()
else:
    dp_degree, dp_rank = 1, 0

if parallel_dims.pp_enabled:
    pp_mesh = world_mesh["pp"]

# Set random seed, and maybe enable deterministic mode (mainly for debugging, expect perf loss)
set_determinism(world_mesh, device, job_config.training.seed,
                job_config.training.deterministic)

data_loader = build_hf_data_loader(
    job_config.training.dataset,
    job_config.training.dataset_path,
    tokenizer,
    job_config.training.batch_size,
    job_config.training.seq_len,
    dp_degree,
    dp_rank,
)


# loss function to be shared by Pipeline Parallel and SPMD training
def loss_fn(pred, labels):
    return torch.nn.functional.cross_entropy(
        pred.flatten(0, 1).float(), labels.flatten(0, 1)
    )

# TODO: compiling loss function causes CUDA errors, turning off for now
# if job_config.training.compile:
#     loss_fn = torch.compile(loss_fn)


INFO:datasets:PyTorch version 2.5.1+rocm6.2 available.
INFO:root:CUDA capacity: AMD Radeon Graphics with 191.98GiB memory
INFO:root:Peak FLOPS used for computing MFU: 3.120e+14


Flash Attention 2.0 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in OPTForCausalLM is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`
Flash Attention 2.0 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in OPTModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`
Flash Attention 2.0 only supports torch.float16 and torch.bfloat16 dtypes, but the current

INFO:root:Preparing c4_test dataset from tests/assets/c4_test
DEBUG:filelock:Attempting to acquire lock 140217978531968 on /workspace/hf_home/datasets/_workspace_hf_home_datasets_c4_test_default_0.0.0_5e9b0fa315c7f3da.lock
DEBUG:filelock:Lock 140217978531968 acquired on /workspace/hf_home/datasets/_workspace_hf_home_datasets_c4_test_default_0.0.0_5e9b0fa315c7f3da.lock
DEBUG:fsspec.local:open file: /workspace/hf_home/datasets/c4_test/default/0.0.0/5e9b0fa315c7f3da/dataset_info.json
DEBUG:filelock:Attempting to release lock 140217978531968 on /workspace/hf_home/datasets/_workspace_hf_home_datasets_c4_test_default_0.0.0_5e9b0fa315c7f3da.lock
DEBUG:filelock:Lock 140217978531968 released on /workspace/hf_home/datasets/_workspace_hf_home_datasets_c4_test_default_0.0.0_5e9b0fa315c7f3da.lock
DEBUG:filelock:Attempting to acquire lock 140079224356640 on /workspace/hf_home/datasets/c4_test/default/0.0.0/5e9b0fa315c7f3da_builder.lock
DEBUG:filelock:Lock 140079224356640 acquired on /workspace/hf_ho

/workspace/transformers/src/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [2]:
parallel_dims

ParallelDims(dp_replicate=1, dp_shard=1, cp=1, tp=1, pp=1, world_size=1, enable_loss_parallel=True)

In [3]:
# a no-op hander if float8 is not enabled
float8_handler = Float8Handler(job_config, parallel_dims)
# swap to Float8Linear based on float8 configs
float8_handler.convert_to_float8_training(model)


In [4]:
# move sharded model to CPU/GPU and initialize weights via DTensor
if job_config.checkpoint.create_seed_checkpoint:
    init_device = "cpu"
    buffer_device = None
elif job_config.training.enable_cpu_offload:
    init_device = "cpu"
    buffer_device = device_type
else:
    init_device = device_type
    buffer_device = None

model.to_empty(device=init_device)
with torch.no_grad():
    reset_params(model)
    model.init_weights()
    model.to(torch.bfloat16)
model.train()
model_parts = [model]

device_mem_stats = device_memory_monitor.get_peak_stats()
logger.info(
    f"{device_type.upper()} memory usage for model: "
    f"{device_mem_stats.max_reserved_gib:.2f}GiB"
    f"({device_mem_stats.max_reserved_pct:.2f}%)"
)

INFO:root:CUDA memory usage for model: 0.81GiB(0.42%)


In [5]:
if job_config.training.compile:
    # for layer_id, transformer_block in model.layers.named_children():
    #     transformer_block = torch.compile(transformer_block, fullgraph=True)
    #     model.layers.register_module(layer_id, transformer_block)
    logger.info("compiling model")
    #torch._dynamo.config.suppress_errors = True
    model = torch.compile(model, fullgraph=True)

In [6]:
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 768, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 768)
      (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-11): 12 x OPTDecoderLayer(
          (self_attn): OptFlashAttention2(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
            (qk): BMM()
            (qk_v): BMM()
            (mult): Mult()
            (softmax): Softmax(dim=-1)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=768, out_features=3072, bias=True)

In [7]:
# build optimizer after applying parallelisms to the model
optimizers = build_optimizers(model_parts, job_config)
lr_schedulers = build_lr_schedulers(optimizers.optimizers, job_config)

train_state = TrainState()

# load initial checkpoint
checkpoint = CheckpointManager(
    dataloader=data_loader,
    model_parts=model_parts,
    optimizers=optimizers,
    lr_schedulers=lr_schedulers,
    states={"train_state": train_state},
    job_config=job_config,
)

checkpoint.load(step=job_config.checkpoint.load_step)
metric_logger = build_metric_logger(job_config, parallel_dims)

# plot losses loaded from checkpoint (if any) to TensorBoard
# NOTE: Loss info after the last log step before checkpoint saving will not be ploted.
#       This can be avoided by setting checkpoint.interval to be a multiple of metrics.log_freq
if train_state.step > 0:
    for idx, step in enumerate(train_state.log_steps):
        metrics = {
            "loss_metrics/global_avg_loss": train_state.global_avg_losses[idx],
            "loss_metrics/global_max_loss": train_state.global_max_losses[idx],
        }
        metric_logger.log(metrics, step=step)

data_iterator = iter(data_loader)

train_context = get_train_context(
    parallel_dims.loss_parallel_enabled,
    job_config.experimental.enable_compiled_autograd,
)

# variables used to keep info for metrics logging
ntokens_since_last_log = 0
data_loading_times = []
time_last_log = time.perf_counter()
device_memory_monitor.reset_peak_stats()
gc_handler = GarbageCollection(gc_freq=job_config.training.gc_freq)

checkpoint.reset()

# train loop
logger.info(
    f"Training starts at step {train_state.step + 1}, "
    f"with local batch size {job_config.training.batch_size}, "
    f"global batch size {job_config.training.batch_size * dp_degree}, "
    f"sequence length {job_config.training.seq_len}, "
    f"total steps {job_config.training.steps} "
    f"(warmup {job_config.training.warmup_steps})"
)
with maybe_enable_profiling(
    job_config, global_step=train_state.step
) as torch_profiler, maybe_enable_memory_snapshot(
    job_config, global_step=train_state.step
) as memory_profiler:
    while train_state.step < job_config.training.steps:
        train_state.step += 1
        gc_handler.run(train_state.step)

        # get batch
        data_load_start = time.perf_counter()
        batch = next(data_iterator)
        input_ids, labels = batch
        ntokens_since_last_log += labels.numel()
        data_loading_times.append(time.perf_counter() - data_load_start)

        input_ids = input_ids.to(device_type)
        labels = labels.to(device_type)
        optimizers.zero_grad()

        # apply context parallelism if cp is enabled
        # ensure CP handles the separate freqs_cis buffer for each pp stage
        optional_context_parallel_ctx = (
            create_context_parallel_ctx(
                cp_mesh=world_mesh["cp"],
                cp_buffers=[input_ids, labels] + [m.freqs_cis for m in model_parts],
                cp_seq_dims=[1, 1] + [0 for _ in model_parts],
                cp_no_restore_buffers={input_ids, labels},
                cp_rotate_method=job_config.experimental.context_parallel_rotate_method,
            )
            if parallel_dims.cp_enabled
            else None
        )

        # Non-PP forward / backward
        with train_context(optional_context_parallel_ctx):
            pred = model(input_ids).logits
            loss = loss_fn(pred, labels)
            # pred.shape=(bs, seq_len, vocab_size)
            # need to free to before bwd to avoid peaking memory
            del pred
            loss.backward()

        # clip gradients
        clip_grad_norm_(
            [p for m in model_parts for p in m.parameters()],
            job_config.training.max_norm,
            foreach=True,
            pp_mesh=pp_mesh if parallel_dims.pp_enabled else None,
        )

        # sync float8 amaxes and scales
        float8_handler.sync_float8_amax_and_scale_history(model_parts)

        # optimizer step
        checkpoint.maybe_wait_for_staging()

        optimizers.step()
        lr_schedulers.step()

        # calculate float8 dynamic amax/scale for all-parameter for FSDP2
        # it issues a single all-reduce for all parameters at once for better performance
        float8_handler.precompute_float8_dynamic_scale_for_fsdp(model_parts)

        # log metrics
        if (
            train_state.step == 1
            or train_state.step % job_config.metrics.log_freq == 0
        ):
            if (
                parallel_dims.dp_replicate_enabled
                or parallel_dims.dp_shard_enabled
                or parallel_dims.cp_enabled
            ):
                loss = loss.detach()
                global_avg_loss, global_max_loss = (
                    dist_mean(loss, world_mesh["dp_cp"]),
                    dist_max(loss, world_mesh["dp_cp"]),
                )
            else:
                global_avg_loss = global_max_loss = loss.item()

            # update train state
            train_state.log_steps.append(train_state.step)
            train_state.global_avg_losses.append(global_avg_loss)
            train_state.global_max_losses.append(global_max_loss)

            time_delta = time.perf_counter() - time_last_log

            # tokens per second per device, abbreviated as tps
            tps = ntokens_since_last_log / (
                time_delta * parallel_dims.non_data_parallel_size
            )

            time_end_to_end = time_delta / job_config.metrics.log_freq
            time_data_loading = sum(data_loading_times) / len(data_loading_times)
            time_data_loading_pct = 100 * sum(data_loading_times) / time_delta

            device_mem_stats = device_memory_monitor.get_peak_stats()

            metrics = {
                "loss_metrics/global_avg_loss": global_avg_loss,
                "loss_metrics/global_max_loss": global_max_loss,
                "throughput(tps)": tps,
                "time_metrics/end_to_end(s)": time_end_to_end,
                "time_metrics/data_loading(s)": time_data_loading,
                "time_metrics/data_loading(%)": time_data_loading_pct,
                "memory/max_active(GiB)": device_mem_stats.max_active_gib,
                "memory/max_active(%)": device_mem_stats.max_active_pct,
                "memory/max_reserved(GiB)": device_mem_stats.max_reserved_gib,
                "memory/max_reserved(%)": device_mem_stats.max_reserved_pct,
                "memory/num_alloc_retries": device_mem_stats.num_alloc_retries,
                "memory/num_ooms": device_mem_stats.num_ooms,
            }
            metric_logger.log(metrics, step=train_state.step)

            logger.info(
                f"{color.cyan}step: {train_state.step:2}  "
                f"{color.green}loss: {global_avg_loss:7.4f}  "
                f"{color.yellow}memory: {device_mem_stats.max_reserved_gib:5.2f}GiB"
                f"({device_mem_stats.max_reserved_pct:.2f}%)  "
                f"{color.blue}tps: {round(tps):,}  "
            )

            ntokens_since_last_log = 0
            data_loading_times.clear()
            time_last_log = time.perf_counter()
            device_memory_monitor.reset_peak_stats()

        checkpoint.save(
            train_state.step, force=(train_state.step == job_config.training.steps)
        )

        # signal the profiler that the next profiling step has started
        if torch_profiler:
            torch_profiler.step()
        if memory_profiler:
            memory_profiler.step()

        # reduce timeout after first train step for faster signal
        # (assuming lazy init and compilation are finished)
        if world_mesh and train_state.step == 1:
            set_pg_timeouts(
                timeout=timedelta(seconds=job_config.comm.train_timeout_seconds),
                world_mesh=world_mesh,
            )

if world_mesh and torch.distributed.get_rank() == 0:
    logger.info("Sleeping 2 seconds for other ranks to complete")
    time.sleep(2)

metric_logger.close()
logger.info("Training completed")

DEBUG:root:Building logger with config: wandb=False, tensorboard=False
DEBUG:root:Logging decision: has_logging_enabled=False, should_log=False
DEBUG:root:Returning BaseLogger due to should_log=False
INFO:root:Training starts at step 1, with local batch size 8, global batch size 8, sequence length 2048, total steps 10 (warmup 2)
INFO:root:step:  1  loss: 450.2670  memory: 14.99GiB(7.81%)  tps: 1,217  
INFO:root:step:  2  loss: 182.0343  memory: 14.99GiB(7.81%)  tps: 37,488  
INFO:root:step:  3  loss: 102.3643  memory: 14.99GiB(7.81%)  tps: 36,367  
INFO:root:step:  4  loss: 97.5003  memory: 14.99GiB(7.81%)  tps: 36,294  
INFO:root:step:  5  loss: 92.9252  memory: 14.99GiB(7.81%)  tps: 36,509  
INFO:root:step:  6  loss: 89.8093  memory: 14.99GiB(7.81%)  tps: 37,638  
INFO:root:step:  7  loss: 86.3709  memory: 14.99GiB(7.81%)  tps: 36,180  
INFO:root:step:  8  loss: 82.0117  memory: 14.99GiB(7.81%)  tps: 37,992  
INFO:root:step:  9  loss: 79.6796  memory: 14.99GiB(7.81%)  tps: 37,535  
I

In [8]:
from transformers.triton_flash_attention_fp8 import _bwd_kernel

print(_bwd_kernel.fn)
kernel = list(_bwd_kernel.fn.device_caches[0][0].values())[0]
#print(print(_bwd_kernel.fn.device_caches[0]))
# print(print(
#     list(_bwd_kernel.fn.device_caches[0][0].values())[0].asm['amdgcn']))
# # _bwd_kernel.fn.compile()
print(kernel.asm.keys())

JITFunction(transformers.triton_flash_attention_fp8:_bwd_kernel)
dict_keys(['ttir', 'ttgir', 'llir', 'amdgcn', 'hsaco'])


In [9]:
from triton.runtime.driver import driver
from triton.compiler import CompiledKernel, compile, ASTSource, make_backend
from triton._C.libtriton import ir, passes, llvm, amd

target = driver.active.get_current_target()
backend = make_backend(target)

stages = dict()
options = backend.parse_options(dict())
metadata = {"target": target}
backend.add_stages(stages, options)
print(stages)


{'ttir': <function HIPBackend.add_stages.<locals>.<lambda> at 0x7f657417b240>, 'ttgir': <function HIPBackend.add_stages.<locals>.<lambda> at 0x7f657417b100>, 'llir': <function HIPBackend.add_stages.<locals>.<lambda> at 0x7f657417b060>, 'amdgcn': <function HIPBackend.add_stages.<locals>.<lambda> at 0x7f657417b380>, 'hsaco': <function HIPBackend.add_stages.<locals>.<lambda> at 0x7f657417b2e0>}


In [10]:
generated_asm = list(_bwd_kernel.fn.device_caches[0][0].values())[0].asm['amdgcn']


def process_string(input_str):
    input_lines = input_str.split("\n")
    output_lines = []
    modified = False

    # Find line ids for each global_atomic_add
    global_atomic_add_ids = [
        i for i, line in enumerate(input_lines)
        if line.strip().startswith("global_atomic_add_f32")
    ]

    if len(global_atomic_add_ids) < 1:
        return input_str, modified

    # Iterate over the line ids
    offset_pre = offset_post = 0
    for idx, line_num in enumerate(global_atomic_add_ids):
        offset_pre = 0
        # if input_lines[line_num - 1].strip().startswith(
        #     ("s_waitcnt", "buffer_wbl2")):
        if input_lines[line_num - 1].strip().startswith("buffer_wbl2"):
            modified = True
            offset_pre += 1
            # if input_lines[line_num - 2].strip().startswith("buffer_inv"):
            #     offset_pre += 1
        if idx == 0:
            output_lines.extend(input_lines[:line_num - offset_pre])
        else:
            output_lines.extend(
                input_lines[global_atomic_add_ids[idx - 1] + offset_post +
                            1:line_num - offset_pre])

        output_lines.append(input_lines[line_num])

        offset_post = 0
        if input_lines[line_num + 1].strip().startswith("s_waitcnt"):
            offset_post += 1
            modified = True
            if input_lines[line_num + 2].strip().startswith("buffer_inv"):
                offset_post += 1

    output_lines.extend(input_lines[line_num + offset_post + 1:])

    output_str = "\n".join(output_lines)
    return output_str, modified

#print(process_string(generated_asm)[0])
print(generated_asm)

	.text
	.amdgcn_target "amdgcn-amd-amdhsa--gfx942"
	.amdhsa_code_object_version 5
	.globl	_bwd_kernel                     ; -- Begin function _bwd_kernel
	.p2align	8
	.type	_bwd_kernel,@function
_bwd_kernel:                            ; @_bwd_kernel
.Lfunc_begin0:
	.cfi_sections .debug_frame
	.cfi_startproc
	s_trap 2 ; Kernarg preload header. Trap with incompatible firmware that doesn't support preloading kernel arguments.
	.fill 63, 4, 0xbf800000 ; s_nop 0
; %bb.0:
	.file	1 "/workspace/transformers/src/transformers" "triton_flash_attention_fp8.py"
	.loc	1 1522 27 prologue_end          ; triton_flash_attention_fp8.py:1522:27
	s_load_dwordx4 s[44:47], s[0:1], 0x90
	s_load_dwordx2 s[60:61], s[0:1], 0xc0
	.loc	1 1525 22                       ; triton_flash_attention_fp8.py:1525:22
	s_mul_hi_i32 s10, s14, 0x2aaaaaab
	s_lshr_b32 s11, s10, 31
	s_ashr_i32 s19, s10, 1
	s_add_i32 s19, s19, s11
	.loc	1 1555 27                       ; triton_flash_attention_fp8.py:1555:27
	s_waitcnt lgkmcnt(0)
	s

In [11]:
loss

tensor(77.2786, device='cuda:0', grad_fn=<NllLossBackward0>)

In [12]:
model.lm_head.weight.grad

tensor([[ 6.0244e-37,  6.7238e-36,  5.0781e-36,  ..., -3.2914e-36,
         -4.1142e-36, -3.8321e-36],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [-2.3007e-05, -2.8229e-04, -2.2411e-04,  ...,  1.3351e-04,
          1.7262e-04,  1.6117e-04],
        ...,
        [ 6.1126e-37,  7.7112e-36,  5.8775e-36,  ..., -3.7146e-36,
         -4.6550e-36, -4.3258e-36],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 2.8800e-37,  3.3619e-36,  2.5743e-36,  ..., -1.5752e-36,
         -2.0219e-36, -1.8573e-36]], device='cuda:0', dtype=torch.bfloat16)

In [13]:
next(model.parameters()).grad

tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 3.8147e-06, -4.5635e-07, -2.8163e-06,  ..., -4.8876e-06,
          6.7428e-07,  5.0068e-06],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00]], device='cuda:0', dtype=torch.bfloat16)

In [14]:
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, 2, foreach=True)

tensor(1., device='cuda:0', dtype=torch.bfloat16)

In [15]:
for n, p in model.named_parameters():
    if p.grad.isnan().any().item():
        print(f"{n}: {p.grad}")

In [16]:
model(input_ids)

CausalLMOutputWithPast(loss=None, logits=tensor([[[  6.8438,   0.0000,  23.6250,  ...,  13.4375,  -9.6250,   9.4375],
         [ 12.0625,   0.0000,  22.1250,  ...,  14.5000,  -2.3281,   7.4375],
         [ 12.0000,   0.0000,  23.7500,  ...,  12.5625,  -4.2812,  14.5625],
         ...,
         [ 11.0000,   0.0000,  28.0000,  ...,  10.8750, -11.5625,  13.8750],
         [ 11.5000,   0.0000,  25.2500,  ...,  12.1875,  -3.6406,  10.5000],
         [  9.1875,   0.0000,  24.1250,  ...,  11.6875,  -5.3750,   9.9375]],

        [[  4.9062,   0.0000,  28.0000,  ...,  11.0000,  -5.4062,   8.2500],
         [ 10.1875,   0.0000,  24.2500,  ...,  10.3125,  -5.0312,   9.9375],
         [ 13.1875,   0.0000,  21.3750,  ...,   9.8750,  -4.2188,  13.7500],
         ...,
         [  8.9375,   0.0000,  25.1250,  ...,   9.0625,  -6.6875,   9.3125],
         [ 14.6250,   0.0000,  24.5000,  ...,  10.8125,  -7.9062,  14.5000],
         [ 11.6250,   0.0000,  26.7500,  ...,  12.5000,  -4.5938,   9.0625]],

   